# Dataset

Police Press releases about traffic accidents in Malta: local_news_articles.csv <br>
Text of local news articles covering accidents: police_press_releases.csv


In [70]:
# Imports
import pandas as pdr
from collections import Counter

In [ ]:
# Police Press dataframe

police_press_df = pd.read_csv("Datasets\police_press_releases.csv")

police_press_df.info()
print('\n')
print(police_press_df.head())

In [ ]:
# Removal of date_modified column on original DF as it is beleived to not be useful.
# Traditional MLs can't use dates as is, will have to convert into YEAR, MONTH, DAY.

police_press_df.drop(columns=["date_modified"], inplace=True)

# Converting to dateTime and slicing date published into YEAR, MONTH, DAY
# Stripping any trailing spaces, blank spaces etc
police_press_df['date_published'] = police_press_df['date_published'].astype(str).str.strip()

police_press_df['date_published'] = (
    pd.to_datetime(
        police_press_df['date_published'].astype(str).str.strip(),
        format='mixed', # Was having trouble with the format even though all rows contain dd/mm/YYYY - specifying to mixed worked.
        dayfirst=True,  # Although specified format to mixed, specifying dayFirst = True should keep the data accurate.
        errors='coerce'
    )
)

police_press_df['year'] = police_press_df['date_published'].dt.year
police_press_df['month'] = police_press_df['date_published'].dt.month
police_press_df['day'] = police_press_df['date_published'].dt.day

# Simply converting the columns into int to remove decimal.
police_press_df['year'] = police_press_df['year'].astype(int)
police_press_df['month'] = police_press_df['month'].astype(int)
police_press_df['day'] = police_press_df['day'].astype(int)


#Dropping date_published column as it is no longer needed
police_press_df.drop(columns=["date_published"], inplace=True)


police_press_df.info()
print('\n')
print(police_press_df.head())

In [ ]:
# For Content / Text.

In [ ]:
# This is to confirm the output of police_press_releases as cleaned dataset

output_file_path_one = 'Datasets\police_press_releases_cleaned.csv'

police_press_df.to_csv(
    output_file_path_one, 
    index=False,
    encoding='utf-8'
)

print(f"Data exported successfully to: {output_file_path_one}")


In [ ]:
# Local News Articles dataframe

local_news_df = pd.read_csv("Datasets\local_news_articles.csv")

local_news_df.info()
print('\n')
print(local_news_df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 321 entries, 0 to 320
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   article_id         321 non-null    int64 
 1   url                321 non-null    object
 2   source_name        321 non-null    object
 3   source_url         321 non-null    object
 4   title              321 non-null    object
 5   subtitle           313 non-null    object
 6   author_name        321 non-null    object
 7   publish_date       321 non-null    object
 8   content            321 non-null    object
 9   top_image_url      318 non-null    object
 10  top_image_caption  312 non-null    object
 11  created_at         321 non-null    object
 12  tags               321 non-null    object
 13  categories         321 non-null    object
dtypes: int64(1), object(13)
memory usage: 35.2+ KB


   article_id                                                url  \
0        4208  https:

In [ ]:
# Removal of: article_id, url, source_url, created_at (publish_date is the relevant field),top_image_url, categories(confirmed to have no values)
# One-hot Encoding: source_name
# Frequency Encoding: author_name

In [72]:
# Confirming only 2 source names are present, one-hot encoding
print('Sources:')
source_name = local_news_df['source_name'].unique()
print(source_name)

# Confirming multiple authors, will be used for the amount of times an author shows up - frequency encoding.
print('\nAuthors:')
author_names = local_news_df['author_name'].unique()
print(author_names)


# Confirming that categories is a completely empty column with empty dictionaries.
print('\nCategories Repr:')
local_news_df['categories'].apply(repr).head()

Sources:
['Times of Malta' 'Newsbook']

Authors:
['Emma Borg' 'Times of Malta' 'Sarah Carabott' 'Matthew Bonanno'
 'Claudia Calleja' 'Edwina Brincat' 'Marc Galdes' 'Neville Borg'
 'Giulia Magri' 'Mark Laurence Zammit' 'Jacob Borg' 'James Cummings'
 'Daniel Ellul' 'Monique Agius' 'Dylan Attard' 'Mariella Cilia'
 'Jurgen Balzan' 'Herman Grech' 'Bertrand Borg']

Categories Repr:


0    '{}'
1    '{}'
2    '{}'
3    '{}'
4    '{}'
Name: categories, dtype: object

In [73]:
# Dropping the columns which are deemed useless
# Removal of: article_id, url, source_url, created_at (publish_date is the relevant field),top_image_url, categories(confirmed to have no values)
local_news_df.drop(columns=["article_id" , "url", "source_url" , "created_at" , "top_image_url", "categories"], inplace=True)

# One-hot encoding Source name
local_news_df = pd.get_dummies(local_news_df, columns=["source_name"], dtype=int)

# Frequency encoding authors
author_counts = local_news_df["author_name"].value_counts()
local_news_df["author_frequency"] = local_news_df['author_name'].map(author_counts)

#Normalizing frequency for ML models.
local_news_df["author_frequency"] = local_news_df["author_frequency"] / len(local_news_df)

#Dropping original column
local_news_df.drop(columns=["author_name"], inplace=True)

local_news_df.info()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 321 entries, 0 to 320
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   title                       321 non-null    object 
 1   subtitle                    313 non-null    object 
 2   publish_date                321 non-null    object 
 3   content                     321 non-null    object 
 4   top_image_caption           312 non-null    object 
 5   tags                        321 non-null    object 
 6   source_name_Newsbook        321 non-null    int32  
 7   source_name_Times of Malta  321 non-null    int32  
 8   author_frequency            321 non-null    float64
dtypes: float64(1), int32(2), object(6)
memory usage: 20.2+ KB


In [74]:
# Slicing the Date Published column into Year, Month, Day as done previously (Copy Pasted code)

local_news_df['publish_date'] = local_news_df['publish_date'].astype(str).str.strip()

local_news_df['publish_date'] = (
    pd.to_datetime(
        local_news_df['publish_date'].astype(str).str.strip(),
        format='mixed',
        dayfirst=True,
        errors='coerce'
    )
)

local_news_df['year'] = local_news_df['publish_date'].dt.year
local_news_df['month'] = local_news_df['publish_date'].dt.month
local_news_df['day'] = local_news_df['publish_date'].dt.day

local_news_df['year'] = local_news_df['year'].astype(int)
local_news_df['month'] = local_news_df['month'].astype(int)
local_news_df['day'] = local_news_df['day'].astype(int)

local_news_df.drop(columns=["publish_date"], inplace=True)

local_news_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 321 entries, 0 to 320
Data columns (total 11 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   title                       321 non-null    object 
 1   subtitle                    313 non-null    object 
 2   content                     321 non-null    object 
 3   top_image_caption           312 non-null    object 
 4   tags                        321 non-null    object 
 5   source_name_Newsbook        321 non-null    int32  
 6   source_name_Times of Malta  321 non-null    int32  
 7   author_frequency            321 non-null    float64
 8   year                        321 non-null    int32  
 9   month                       321 non-null    int32  
 10  day                         321 non-null    int32  
dtypes: float64(1), int32(5), object(5)
memory usage: 21.4+ KB


In [76]:
# For tags, I firstly turn the dicts into a single flattened list of all unique tags.
# The help of ChatGPT was utilized here.

# Step 1: Parse the tags manually
def parse_tags_manual(x):
    if not isinstance(x, str) or x.strip() == "":
        return []
    x = x.strip('{}')
    tags = [tag.strip().strip('"').strip("'") for tag in x.split(',')]
    return [tag for tag in tags if tag]

tags_lists = local_news_df['tags'].apply(parse_tags_manual)

# Step 2: Flatten all tags into a single list
all_tags = [tag for sublist in tags_lists for tag in sublist]

# Step 3: Count frequencies
tag_counts = Counter(all_tags)

# Step 4: Convert to a sorted DataFrame for readability
tag_freq_df = pd.DataFrame(tag_counts.items(), columns=['tag', 'frequency']).sort_values(by='frequency', ascending=False)

# Display the frequencies
print(tag_freq_df)

# Step 5: Get top 5 tags
top_tags = tag_freq_df['tag'].head(5).tolist()
print("Top 5 tags:", top_tags)

# Step 6: One-hot encode top tags using Pandas only
for tag in top_tags:
    local_news_df[f"tag_{tag}"] = tags_lists.apply(lambda x: int(tag in x))

# Step 7: Drop original tags column if no longer needed
local_news_df.drop(columns=['tags'], inplace=True)

# Inspect
print(local_news_df.head())




                       tag  frequency
2                 National        313
0                 Accident        222
7                  Traffic        142
15                  Police         69
17               Transport         31
..                     ...        ...
47               Ombudsman          1
88                 Buġibba          1
90                  Diving          1
91                    Tech          1
127  Hotels/Hostels/Airbnb          1

[128 rows x 2 columns]
Top 5 tags: ['National', 'Accident', 'Traffic', 'Police', 'Transport']
                                               title  \
0  Driver stuck in traffic says speeding LESA car...   
1  PN slams government for diverting EU bus funds...   
2  Motorcyclist seriously hurt in St Paul's Bay b...   
3  Skip involved in horror St Paul’s Bay bypass c...   
4  Two people, including teenage girl, critically...   

                                            subtitle  \
0  ‘I was shocked at that moment but more so frus...   


In [ ]:
# For Content / Text.

In [77]:
output_file_path_two = 'Datasets\local_news_articles_cleaned.csv'


local_news_df.to_csv(
    output_file_path_two, 
    index=False,
    encoding='utf-8'
)

print(f"Data exported successfully to: {output_file_path_two}")

Data exported successfully to: Datasets\local_news_articles_cleaned.csv


# We are to add on top of the given datasets

## 1. Weather / Geographical APIs
- **Open-Meteo**  
  Free weather API with global coverage. Supports forecast, historical data, and climate data.  
  [https://open-meteo.com](https://open-meteo.com)

- **Copernicus Climate Data Source**  
  Provides free access to historical climate data (temperature, precipitation, etc.) globally.  
  [https://climate.copernicus.eu](https://climate.copernicus.eu)

---

## 2. Temporal Data

### Holidays
- **Nager.Date**  
  Free REST API for public holidays worldwide. No authentication required.  
  [https://date.nager.at/](https://date.nager.at/)

- **OpenHolidays API**  
  Provides public and school holidays. JSON responses, no auth needed.  
  [https://www.openpublicapis.com/api/openholidays-api](https://www.openpublicapis.com/api/openholidays-api)

---

### Free Traffic / Rush Hour Data
- **Open Traffic / OTv2**  
  Open-source platform for historical and real-time traffic/mobility data. Useful to infer rush-hour patterns.  
  [https://github.com/opentraffic/otv2-platform](https://github.com/opentraffic/otv2-platform)

- **GTFS (Static GTFS)**  
  Provides transit schedules, useful to infer expected peak travel periods (rush hours).  
  [https://developers.google.com/transit/gtfs](https://developers.google.com/transit/gtfs)

- **TomTom Traffic API (Free Tier)**  
  Limited free tier for traffic flow and congestion data. Can be used to estimate rush hours.  
  [https://developer.tomtom.com/traffic-api](https://developer.tomtom.com/traffic-api)